<a href="https://colab.research.google.com/github/rd908213/ME-597-IIoT/blob/main/ml_tutorial/HW/HW4/HW4_robertclaud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# ML Homework 4 Guide


1. Save a copy of this ipynb file in your GoogleDrive or PC.
2. Edit the name of this code from "HW4.ipynb" to "HW4\_(your name).ipynb"
3. Fill out the code cells below according to descriptions.
4. Save and upload to BrightSpace. (DO NOT clear the outputs of your code)
5. Convert the .ipynb file to PDF file and upload together
6. Upload the saved confusion matrix image file together


# [HW 4] Grid search for CNN

1. Load the 'AccData.csv' file as you loaded in the 'ML7_Code2'.
2. Prepare training and test dataset with the test data ratio of """30%""".
3. Perform a grid search to find the best combination of hyperparameters for the CNN model.

- You can create any combination of hyperparameters.
- Do not copy 'ML7_Code2' exactly. Set at least one other hyperparamter as the grid search target.

4. Determine your best CNN model based on classification accuracy.
5. Plot a confusion matrix for the best model and save it as an image file (.png or .jpg).

- Search how to save a figure as an image file.
- Upload the confusion matrix image to BrightSpace with ipynb and pdf files


### Import Packages


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Import 'Tensorflow' pakage
import tensorflow as tf
from tensorflow import keras
from scipy import signal
from sklearn.model_selection import train_test_split

In [ ]:
# Access to google drive
from google.colab import drive

drive.mount("/content/drive")

### Load the 'AccData.csv' File from ML7_Code2


In [ ]:
AccData_pd_load = pd.read_csv(
    "https://github.com/purduelamm/purdue_me597_iiot/blob/main/ml_tutorial/Dataset_Acc/AccData.csv?raw=true"
).iloc[:, 1:]
AccData_pd_load.shape
AccData = np.array(AccData_pd_load)
AccData.shape

### Prepare Training and Test Dataset with the test data ratio of 30%


In [ ]:
# Convert to Spectogram by STFT
Fs = 12800  # Sampling Frequency
f, t, AccSTFT = signal.spectrogram(AccData, Fs, nperseg=78, noverlap=10)
AccSTFT.shape

# Split Training & Test Data
NoOfData = 180

NormalSet = AccSTFT[:NoOfData]
AbnormalSet = AccSTFT[NoOfData:]

NoOfSensor = 1
NormalSet = NormalSet.reshape(
    NormalSet.shape[0], NormalSet.shape[1], NormalSet.shape[2], NoOfSensor
)
AbnormalSet = AbnormalSet.reshape(
    AbnormalSet.shape[0],
    AbnormalSet.shape[1],
    AbnormalSet.shape[2],
    NoOfSensor,
)

NormalSet.shape, AbnormalSet.shape

# Designate test data ratio
TestData_Ratio = 0.3  # THIS IS WHERE THE RATIO IS SET TO 30%

TrainData_Nor, TestData_Nor = train_test_split(
    NormalSet, test_size=TestData_Ratio, random_state=777
)
TrainData_Abn, TestData_Abn = train_test_split(
    AbnormalSet, test_size=TestData_Ratio, random_state=777
)

print(TrainData_Nor.shape, TestData_Nor.shape)
print(TrainData_Abn.shape, TestData_Abn.shape)

#### Data Labeling and Preparation


In [ ]:
TrainLabel_Nor = np.zeros((TrainData_Nor.shape[0], 2))
TrainLabel_Abn = np.ones((TrainData_Abn.shape[0], 2))
TestLabel_Nor = np.zeros((TestData_Nor.shape[0], 2))
TestLabel_Abn = np.ones((TestData_Abn.shape[0], 2))

TrainLabel_Nor[:, 0] = 1  # [1,0]: Normal
TrainLabel_Abn[:, 0] = 0  # [0,1]: Abnormal
TestLabel_Nor[:, 0] = 1  # [1,0]: Normal
TestLabel_Abn[:, 0] = 0  # [0,1]: Abnormal

print(TrainLabel_Nor.shape, TestLabel_Nor.shape)
print(TrainLabel_Abn.shape, TestLabel_Abn.shape)

TrainData = np.concatenate([TrainData_Nor, TrainData_Abn], axis=0)
TestData = np.concatenate([TestData_Nor, TestData_Abn], axis=0)
TrainLabel = np.concatenate([TrainLabel_Nor, TrainLabel_Abn], axis=0)
TestLabel = np.concatenate([TestLabel_Nor, TestLabel_Abn], axis=0)

print(TrainData.shape, TestData.shape)
print(TrainLabel.shape, TestLabel.shape)

### Hyperparameters with at least one different target from ML7_Code2


In [ ]:
# Hyperparameters for grid search
param_FiltS = [3, 5]  # filter(kernel) size (only convolution layer)
param_FiltN = [2, 4]  # number of filters   (only convolution layer)
param_Strid = [1, 2]  # stride              (only convolution layer)

# Fixed hyperparameters
noOfNeuron = 10
learningRate = 0.0001
Epoch = 1000

# Calculate the number of cases
NoOfCases = len(param_FiltS) * len(param_FiltN) * len(param_Strid)
NoOfCases

### CNN Model Definition


In [ ]:
def CNN_model(
    input_data, noOfNeuron, learningRate, filterSize, numOfFilters, stride
):
    model = keras.Sequential()
    model.add(keras.layers.Input(shape=input_data.shape[1:]))
    model.add(
        keras.layers.Conv2D(
            filters=numOfFilters,
            kernel_size=(filterSize, filterSize),
            strides=(stride, stride),
            activation="relu",
            padding="same",
        )
    )
    model.add(keras.layers.MaxPooling2D(pool_size=(2, 2)))
    model.add(
        keras.layers.Conv2D(
            filters=numOfFilters * 2,
            kernel_size=(filterSize, filterSize),
            strides=(1, 1),
            activation="relu",
            padding="same",
        )
    )
    model.add(keras.layers.MaxPooling2D(pool_size=(2, 2)))
    model.add(keras.layers.Flatten())
    model.add(keras.layers.Dense(noOfNeuron, activation="relu"))
    model.add(keras.layers.Dense(2, activation="softmax"))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learningRate),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

In [ ]:
# Create an empty dataframe to store the accuracy results
Accuracy_df = pd.DataFrame(
    np.zeros(shape=(NoOfCases, 4)),
    columns=["filter size", "number of filters", "stride", "Accuracy"],
)
Accuracy_df

### Training with different HyperParameters


In [ ]:
import itertools

# Initialize a count value to store the performance of each model
cnt = 0

# Iterate through all combinations of hyperparameters
for filtS, filtN, strid in itertools.product(
    param_FiltS, param_FiltN, param_Strid
):
    # Create and train the model
    model = CNN_model(TrainData, noOfNeuron, learningRate, filtS, filtN, strid)
    model.fit(TrainData, TrainLabel, epochs=Epoch, verbose=0)

    # Evaluate on test data
    test_loss, test_accuracy = model.evaluate(TestData, TestLabel, verbose=0)

    # Store results in dataframe
    Accuracy_df.loc[cnt] = [filtS, filtN, strid, test_accuracy]
    cnt += 1

# Display the dataframe
Accuracy_df

#### Confirm the grid search results

In [ ]:
# Sort the Accuracy_df by 'Accuracy' column in descending order
Accuracy_df_sorted = Accuracy_df.sort_values(by='Accuracy', ascending=False).reset_index(drop=True)

# Output the best case
Best_FiltS = int(Accuracy_df_sorted.iloc[0, 0])
Best_FiltN = int(Accuracy_df_sorted.iloc[0, 1])
Best_Strid = int(Accuracy_df_sorted.iloc[0, 2])

print(f"[Best case]\n" +
      f"Filter size   : [{Best_FiltS},{Best_FiltS}]\n" +
      f"Num of Filters: {Best_FiltN}\n" +
      f"Strides       : {Best_Strid}\n" +
      "Accuracy: %.2f" % (Accuracy_df_sorted.iloc[0, 3]))

# Calculate mean and standard deviation accuracy for each filter size
mean_accuracy_FiltS = Accuracy_df.groupby(['filter size'])['Accuracy'].agg(['mean', 'std']).reset_index()
mean_accuracy_FiltS

# Calculate mean and standard deviation of accuracy for each number of filter
mean_accuracy_FiltN = Accuracy_df.groupby(['number of filters'])['Accuracy'].agg(['mean', 'std']).reset_index()
mean_accuracy_FiltN

# Calculate mean and standard deviation of accuracy for each stride
mean_accuracy_Strid = Accuracy_df.groupby(['stride'])['Accuracy'].agg(['mean', 'std']).reset_index()
mean_accuracy_Strid

#### Confusion Matrix

In [ ]:
# Retrieve activation function, hidden layers, and learning rate values from the first row of 'Accuracy_df_sorted'
Best_FiltS = int(Accuracy_df_sorted.iloc[0, 0])
Best_FiltN = int(Accuracy_df_sorted.iloc[0, 1])
Best_Strid = int(Accuracy_df_sorted.iloc[0, 2])

# Load the best ANN model using the retrieved hyperparameters
best_cnn_model_name = f'CNN_FS{Best_FiltS}_FN{Best_FiltN}_St{Best_Strid}.h5'
best_cnn_model = keras.models.load_model('/content/drive/MyDrive/Colab Notebooks/SavedFiles/ML_Models/GridSearch_CNN/' + best_cnn_model_name)

# Predict the output (Robotic spot-welding condition) for the test data
Predicted = best_cnn_model.predict(TestData)

# Convert TestLabel and Predicted into vectors to calculate the confusion matrix and evaluation metrics
TestLabel_rev = np.argmax(TestLabel, axis=1)
Predicted_rev = np.argmax(Predicted, axis=1)

# Plot the confusion matrix
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Calculate the confusion matrix
cm = confusion_matrix(TestLabel_rev, Predicted_rev)

plt.figure(figsize=(6, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap=plt.cm.Blues, cbar=False, square=True)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix of the Best CNN Model")
plt.show()

#### Evaluation Metrics

In [ ]:
from sklearn import metrics

# Calculate the evaluation metrics
accuracy  = metrics.accuracy_score(TestLabel_rev, Predicted_rev)
precision = metrics.precision_score(TestLabel_rev, Predicted_rev)
recall    = metrics.recall_score(TestLabel_rev, Predicted_rev)
f1_score  = metrics.f1_score(TestLabel_rev, Predicted_rev)

# Print the evaluation metrics
print(f"Best CNN Model Evaluation:\n")
print(f"Accuracy : {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall   : {recall:.2f}")
print(f"F1 Score : {f1_score:.2f}")

## ML7 and ML8 Summary and Deliverables

Answer the following questions for your achievements


### Q1. Please summarize ML7 and ML8.

---

Write down A1 here.

---


### Q2. What skills did you have to develop to accomplish this project?

---

Wirte down A2 here.

---


### Q3. What aspects of this project were the most beneficial for your learning?

---

Wirte down A3 here.

---


### Q4. What challenges did you encounter in completing the project?

---

Wirte down A4 here.

---


### Q5. How did you overcome the challenges or remedy the problems encountered?

---

Wirte down A5 here.

---
